In [ ]:
# Ячейка 1: Загрузка данных и разделение выборки
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Загрузите ваш файл в Colab (через меню "Файлы" или Google Диск)
df = pd.read_csv('dispensarization_data_2026.csv')
print(f"Размер данных: {df.shape}")
print(f"Целевая переменная 'Доклинический_риск':\n{df['Доклинический_риск'].value_counts()}")

target = 'Доклинический_риск'
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Размер данных: (1000, 18)
Целевая переменная 'Доклинический_риск':
Доклинический_риск
0    939
1     61
Name: count, dtype: int64
Train: (800, 17), Test: (200, 17)


In [ ]:
# Ячейка 2: Функция для расчёта метрик
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

def evaluate_model(model, X_test, y_test, model_name):
    y_pred = model.predict(X_test)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred
    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba)
    }
    print(f"\n{model_name} метрики:")
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics

In [ ]:
# Ячейка 3: FLAML (Microsoft)
!pip install flaml[automl] --quiet

import time
from flaml import AutoML

start = time.time()
automl_flaml = AutoML()
automl_flaml.fit(X_train, y_train, task="classification", time_budget=120,
                 eval_method='holdout', metric='roc_auc', seed=42, verbose=0)
time_flaml = time.time() - start

metrics_flaml = evaluate_model(automl_flaml, X_test, y_test, "FLAML")
metrics_flaml['time_sec'] = time_flaml

# Сохраним результаты во временный файл, чтобы не потерять после перезапуска
import pickle
with open('results_flaml.pkl', 'wb') as f:
    pickle.dump(metrics_flaml, f)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.7/349.7 kB 19.5 MB/s eta 0:00:00


INFO:flaml.tune.searcher.blendsearch:No low-cost partial config given to the search algorithm. For cost-frugal search, consider providing low-cost values for cost-related hps via 'low_cost_partial_config'. More info can be found at https://microsoft.github.io/FLAML/docs/FAQ#about-low_cost_partial_config-in-tune



FLAML метрики:
  accuracy: 0.9700
  precision: 1.0000
  recall: 0.5000
  f1: 0.6667
  roc_auc: 1.0000


In [ ]:
# Ячейка 4: CatBoost (AutoML на основе автоматического поиска)
!pip install catboost -q

import catboost as cb
from catboost import CatBoostClassifier
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import time
import numpy as np

# Определяем сетку гиперпараметров для автоматического подбора
param_grid = {
    'iterations': [100, 300, 500],
    'depth': [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3, 5, 7],
    'border_count': [32, 64, 128]
}

# Используем RandomSearchCV (быстрее полного перебора) для AutoML-поиска
catboost_auto = CatBoostClassifier(
    verbose=0,          # отключаем вывод каждого шага
    random_seed=42,
    cat_features=None,  # ваши признаки уже числовые, но можно указать категориальные, если есть
    auto_class_weights='Balanced'  # автоматическая балансировка весов классов
)

# Поиск гиперпараметров с кросс-валидацией (AutoML)
random_search = RandomizedSearchCV(
    catboost_auto,
    param_distributions=param_grid,
    n_iter=20,           # количество случайных комбинаций
    cv=5,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1,
    verbose=0
)

start = time.time()
random_search.fit(X_train, y_train)
print(f"CatBoost AutoML (RandomizedSearchCV) обучен за {time.time()-start:.2f} сек")

# Лучшая модель
best_catboost = random_search.best_estimator_
print("Лучшие параметры CatBoost:", random_search.best_params_)

# Предсказания
y_pred = best_catboost.predict(X_test)
y_proba = best_catboost.predict_proba(X_test)[:, 1]

# Оценка метрик
metrics = {
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'roc_auc': roc_auc_score(y_test, y_proba)
}
print("\nCatBoost (AutoML) метрики:")
for k, v in metrics.items():
    print(f" {k}: {v:.4f}")

CatBoost AutoML (RandomizedSearchCV) обучен за 164.54 сек
Лучшие параметры CatBoost: {'learning_rate': 0.1, 'l2_leaf_reg': 3, 'iterations': 500, 'depth': 4, 'border_count': 64}

CatBoost (AutoML) метрики:
 accuracy: 0.9850
 precision: 0.8462
 recall: 0.9167
 f1: 0.8800
 roc_auc: 0.9982


1.FLAML:

Преимущества: очень высокая точность (precision = 1.0), отличный ROC AUC (1.0), быстрое обучение.

Недостаток: критически низкий recall (0.5) – модель пропускает половину реальных положительных случаев. Для медицинской задачи это неприемлемо, так как невыявленный риск может привести к серьёзным последствиям.

2.CatBoost:

Преимущества: сбалансированные метрики, recall 0.9167 (обнаруживает 11 из 12 пациентов с риском), F1 = 0.88. Несмотря на меньшую precision (0.8462), модель надёжнее выявляет целевую группу.

Недостаток: большее время обучения (~2.7 минуты), однако оно остаётся приемлемым для статического набора данных.

ROC AUC у обеих моделей близок к 1.0, что говорит об отличной разделяющей способности. Однако для FLAML идеальное значение AUC при low recall – признак того, что модель хорошо ранжирует объекты, но порог классификации выбран неудачно (вероятно, из за дисбаланса и дефолтных настроек).

Итог:

Из двух рассмотренных AutoML‑подходов CatBoost обеспечивает более практичный баланс между полнотой и точностью, что делает его лучшим выбором для задачи выявления доклинического риска. FLAML быстрее и проще, но требует дополнительной настройки для работы с сильным дисбалансом классов.
